In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/val_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/train_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/test_ml_clean_text.csv


In [2]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("/kaggle/input/datasets/kimanaru/dataset-tien-xu-li")

train_df = pd.read_csv(RESULT_DIR / "train_ml_clean_text.csv")
val_df = pd.read_csv(RESULT_DIR / "val_ml_clean_text.csv")
test_df = pd.read_csv(RESULT_DIR / "test_ml_clean_text.csv")

train_df["ml_clean_text"] = train_df["ml_clean_text"].fillna("").astype(str)
val_df["ml_clean_text"] = val_df["ml_clean_text"].fillna("").astype(str)
test_df["ml_clean_text"] = test_df["ml_clean_text"].fillna("").astype(str)

train_df["category"] = train_df["category"].astype(str)
val_df["category"] = val_df["category"].astype(str)
test_df["category"] = test_df["category"].astype(str)

X_train = train_df["ml_clean_text"]
y_train = train_df["category"]

X_val = val_df["ml_clean_text"]
y_val = val_df["category"]

X_test = test_df["ml_clean_text"]
y_test = test_df["category"]

# Thử các model

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import time
import joblib

BOW_MAX_FEATURES = 50000
BOW_NGRAM_RANGE = (1, 2)

RANDOM_STATE = 42
CLASS_WEIGHT = "balanced"
N_JOBS = -1

n_estimators_list = [50, 100, 150, 200]
max_depth_list = [20, 30, 50]
min_samples_split_list = [2, 5]

In [4]:
results = []
best_score = -1
best_model = None
best_params = None

total_runs = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(min_samples_split_list)
)

run_id = 1

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        for min_samples_split in min_samples_split_list:

            print("=" * 80)
            print(f"Running {run_id}/{total_runs}")
            print("n_estimators:", n_estimators)
            print("max_depth:", max_depth)
            print("min_samples_split:", min_samples_split)

            model = Pipeline([
                ("bow", CountVectorizer(
                    max_features=BOW_MAX_FEATURES,
                    ngram_range=BOW_NGRAM_RANGE,
                    lowercase=False
                )),
                ("rf", RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    class_weight=CLASS_WEIGHT,
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS
                ))
            ])

            start = time.time()

            model.fit(X_train, y_train)

            y_val_pred = model.predict(X_val)
            val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")

            elapsed = time.time() - start

            print("Validation Macro F1:", val_macro_f1)
            print("Time:", round(elapsed, 2), "seconds")

            current_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_split": min_samples_split
            }

            results.append({
                "feature_type": "BoW",
                "max_features": BOW_MAX_FEATURES,
                "ngram_range": BOW_NGRAM_RANGE,
                **current_params,
                "class_weight": CLASS_WEIGHT,
                "random_state": RANDOM_STATE,
                "val_macro_f1": val_macro_f1,
                "time_seconds": elapsed
            })

            if val_macro_f1 > best_score:
                best_score = val_macro_f1
                best_model = model
                best_params = current_params
                print("New best Random Forest!")

            run_id += 1

results_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)

print("=" * 80)
print("BEST RANDOM FOREST PARAMS:", best_params)
print("BEST VALIDATION MACRO F1:", best_score)

results_df

Running 1/24
n_estimators: 50
max_depth: 20
min_samples_split: 2
Validation Macro F1: 0.49866832863015703
Time: 9.3 seconds
New best Random Forest!
Running 2/24
n_estimators: 50
max_depth: 20
min_samples_split: 5
Validation Macro F1: 0.4993267518781703
Time: 9.12 seconds
New best Random Forest!
Running 3/24
n_estimators: 50
max_depth: 30
min_samples_split: 2
Validation Macro F1: 0.5198047516246311
Time: 10.49 seconds
New best Random Forest!
Running 4/24
n_estimators: 50
max_depth: 30
min_samples_split: 5
Validation Macro F1: 0.5203325851821127
Time: 9.75 seconds
New best Random Forest!
Running 5/24
n_estimators: 50
max_depth: 50
min_samples_split: 2
Validation Macro F1: 0.5433609405441463
Time: 15.49 seconds
New best Random Forest!
Running 6/24
n_estimators: 50
max_depth: 50
min_samples_split: 5
Validation Macro F1: 0.54445515550561
Time: 12.26 seconds
New best Random Forest!
Running 7/24
n_estimators: 100
max_depth: 20
min_samples_split: 2
Validation Macro F1: 0.5238518088376404
Time:

,feature_type,max_features,ngram_range,n_estimators,max_depth,min_samples_split,class_weight,random_state,val_macro_f1,time_seconds
22,BoW,50000,"(1, 2)",200,50,2,balanced,42,0.564618,35.965902
16,BoW,50000,"(1, 2)",150,50,2,balanced,42,0.562960,28.656606
23,BoW,50000,"(1, 2)",200,50,5,balanced,42,0.562200,24.611189
17,BoW,50000,"(1, 2)",150,50,5,balanced,42,0.561703,20.133845
10,BoW,50000,"(1, 2)",100,50,2,balanced,42,0.559866,22.159494
11,BoW,50000,"(1, 2)",100,50,5,balanced,42,0.559591,16.470626
20,BoW,50000,"(1, 2)",200,30,2,balanced,42,0.546557,18.853059
21,BoW,50000,"(1, 2)",200,30,5,balanced,42,0.545707,14.727353
5,BoW,50000,"(1, 2)",50,50,5,balanced,42,0.544455,12.258646
4,BoW,50000,"(1, 2)",50,50,2,balanced,42,0.543361,15.491511


In [5]:
y_test_pred = best_model.predict(X_test)

print("Best Random Forest params:", best_params)
print("Test Macro F1:", f1_score(y_test, y_test_pred, average="macro"))
print(classification_report(y_test, y_test_pred))

Best Random Forest params: {'n_estimators': 200, 'max_depth': 50, 'min_samples_split': 2}
Test Macro F1: 0.5691269125188246
                precision    recall  f1-score   support

  BLACK VOICES       0.42      0.47      0.45       687
      BUSINESS       0.48      0.53      0.50       899
        COMEDY       0.56      0.44      0.50       810
 ENTERTAINMENT       0.54      0.55      0.54      2605
  FOOD & DRINK       0.60      0.78      0.68       951
HEALTHY LIVING       0.23      0.32      0.26      1004
 HOME & LIVING       0.56      0.70      0.62       648
     PARENTING       0.50      0.60      0.54      1319
       PARENTS       0.33      0.27      0.29       593
      POLITICS       0.88      0.71      0.78      5341
  QUEER VOICES       0.72      0.71      0.72       952
        SPORTS       0.59      0.69      0.64       761
STYLE & BEAUTY       0.68      0.80      0.74      1472
        TRAVEL       0.67      0.67      0.67      1485
      WELLNESS       0.66      0.56